In [27]:
# Notebook 02 - LIMPEZA E PRÉ-PROCESSAMENTO

import os
import pandas as pd

# 1. Carregar o manifesto gerado no Notebook 01
caminho_manifesto = "../data/interim/df_manifesto.csv"
if not os.path.exists(caminho_manifesto):
  raise FileNotFoundError(
      "Execute o Notebook 01 primeiro para gerar o manifesto de arquivos!"
  )

df_manifesto = pd.read_csv(caminho_manifesto)

# 2. Inicializar lista para acumular os dados limpos
dados_limpos_lista = []

# 3. Iterar sobre cada arquivo do manifesto
for _, row in df_manifesto.iterrows():
  sujeito_id = row["sujeito_id"]
  nome_arquivo = row["nome_arquivo"]
  caminho = row["caminho_completo"]

  df_temp = None

  # Carregamento inteligente com suporte a separadores diferentes para CSV
  try:
    if caminho.endswith(".csv"):
      # Tenta ler com separador ';' primeiro (comum em exports em pt-BR), senão tenta ','
      try:
        df_temp = pd.read_csv(
            caminho, sep=";", on_bad_lines="skip", engine="python"
        )
      except Exception:
        df_temp = pd.read_csv(
            caminho, sep=",", on_bad_lines="skip", engine="python"
        )
    elif caminho.endswith(".xlsx"):
      df_temp = pd.read_excel(caminho)
    else:
      continue
  except Exception as e:
    print(f"Aviso: Não foi possível processar o arquivo {caminho}. Erro: {e}")
    continue

  if df_temp is None or df_temp.empty:
    print(f"Aviso: Arquivo vazio ou ilegível: {nome_arquivo}")
    continue

  # 4. Ajuste dinâmico para múltiplas variações de nomenclatura de Gaze
  colunas_possiveis = [
      ["Gaze X", "Gaze Y"],
      ["Gaze_X", "Gaze_Y"],
      ["GazeX", "GazeY"],
      ["X", "Y"],
  ]
  colunas_gaze = None

  for cols in colunas_possiveis:
    if all(col in df_temp.columns for col in cols):
      colunas_gaze = cols
      break

  if colunas_gaze:
    # Remove nulos nas colunas de Gaze identificadas
    df_temp = df_temp.dropna(subset=colunas_gaze)
    # Padroniza sempre para 'GazeX' e 'GazeY'
    df_temp = df_temp.rename(
        columns={colunas_gaze[0]: "GazeX", colunas_gaze[1]: "GazeY"}
    )
  else:
    print(f"--- ATENÇÃO NO ARQUIVO: {nome_arquivo} ---")
    print(f"Colunas disponíveis encontradas: {list(df_temp.columns)}")
    print("-" * 50)
    continue

  # Adicionar identificador do sujeito padronizado
  df_temp["sujeito_id"] = sujeito_id

  # Definir rótulo alvo (Target / y) binário: 0 para Controle, 1 para TEA
  # Verificamos se o nome do arquivo contém 'CONTROLE' para rotular corretamente
  if "CONTROLE" in nome_arquivo.upper():
    df_temp["target"] = 0
  else:
    df_temp["target"] = 1

  # 5. Criar janelamento temporal base (blocos de 1000 amostras)
  tamanho_janela = 1000
  df_temp["janela_id"] = [i // tamanho_janela for i in range(len(df_temp))]

  dados_limpos_lista.append(df_temp)

# Consolidar todos os dados limpos em um único DataFrame intermediário
if len(dados_limpos_lista) > 0:
  df_interim = pd.concat(dados_limpos_lista, ignore_index=True)
  print(f"\nDataset intermediário consolidado com sucesso! Linhas totais: {len(df_interim)}")
  print(f"Sujeitos incluídos: {df_interim['sujeito_id'].unique()}")
  print(f"Distribuição de classes (target):")
  print(df_interim[["sujeito_id", "target"]].drop_duplicates()["target"].value_counts())

# 6. Salvar dados limpos e preparados em formato CSV seguro na pasta interim
os.makedirs("../data/interim", exist_ok=True)
caminho_saida = "../data/interim/df_interim_limpo.csv"

df_interim.to_csv(caminho_saida, index=False)
print(f"\nDados salvos com sucesso em '{caminho_saida}'.")


Dataset intermediário consolidado com sucesso! Linhas totais: 440041
Sujeitos incluídos: <StringArray>
['T47', 'T49', 'T40', 'T30', 'T16', 'T27', 'T55', 'T37', 'T20', 'T58', 'T22',
 'T51', 'T48', 'T19', 'T28', 'T39', 'T35', 'T44', 'T59', 'T34', 'T56', 'T60',
 'T41', 'T32', 'T26', 'T24', 'T38', 'T36', 'T57', 'T54', 'T21', 'T43', 'T18',
 'T17', 'T25', 'T29', 'T46', 'T50', 'T15', 'T61', 'T12']
Length: 41, dtype: str
Distribuição de classes (target):
target
0    23
1    18
Name: count, dtype: int64

Dados salvos com sucesso em '../data/interim/df_interim_limpo.csv'.
